# 01 — Leakage study on BreakHis (run in Google Colab)

**Question:** how much does accuracy drop when we split by *patient* instead of by *image*?

Before you start: **Runtime → Change runtime type → T4 GPU**, then run the cells top to bottom.
Results are saved to your Google Drive, so a disconnect doesn't lose finished runs — just re-run and it skips what's done.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
from google.colab import drive
drive.mount('/content/drive')
RESULTS = '/content/drive/MyDrive/breakhis-results'

## 1. Get the code

In [ ]:
import os, sys
REPO = '/content/breakhis-reliability'
if not os.path.exists(REPO):
    !git clone -q https://github.com/amubakekengbam/breakhis-reliability.git {REPO}
else:
    !git -C {REPO} pull -q
%cd {REPO}
!pip install -q -e .
# Colab doesn't see an editable install until a restart, so add the source folder directly.
if f'{REPO}/src' not in sys.path:
    sys.path.insert(0, f'{REPO}/src')
import breakhis; print('breakhis', breakhis.__version__, 'ready')

## 2. Download BreakHis (~4–5 GB, about 15–20 minutes the first time)
From the official UFPR page. The authors ask you to fill in a short form after downloading and to cite
Spanhol et al. (2016) in any publication. Non-commercial research use only.

**If it stops halfway, just run the cell again — it resumes.** Keep the browser tab open while it downloads.

In [ ]:
# Download BreakHis (~4-5 GB). Saved on the fast local Colab disk, and RESUMES if it was interrupted.
# Every time you re-run this cell it checks the file is complete before unpacking.
import os, subprocess
URL = 'http://www.inf.ufpr.br/vri/databases/BreaKHis_v1.tar.gz'
ARCHIVE = '/content/BreaKHis_v1.tar.gz'
DATA_DIR = '/content/BreaKHis_v1'

def remote_size(url):
    out = subprocess.run(['curl', '-sIL', url], capture_output=True, text=True).stdout
    sizes = [int(l.split(':')[1]) for l in out.lower().splitlines() if l.startswith('content-length')]
    return sizes[-1] if sizes else None

if os.path.exists(f'{DATA_DIR}/histology_slides'):
    print('Data already unpacked - nothing to do.')
else:
    expected = remote_size(URL)
    print('Expected size:', f'{expected/1e9:.2f} GB' if expected else 'unknown')
    !wget -c -q --show-progress -O {ARCHIVE} {URL}
    have = os.path.getsize(ARCHIVE)
    print(f'Downloaded: {have/1e9:.2f} GB')
    if expected and have != expected:
        raise RuntimeError('Download incomplete - just run this cell again; it will resume where it stopped.')
    print('Unpacking (a few minutes)...')
    !tar -xzf {ARCHIVE} -C /content/
    print('Done:', os.path.exists(f'{DATA_DIR}/histology_slides'))

## 3. Check the data matches the paper (7,909 images, 82 patients: 24 benign, 58 malignant)

In [ ]:
from breakhis.data import build_metadata, summarize
df = build_metadata(DATA_DIR)
print(len(df), 'images')
summarize(df)

## 4. Run the experiment
Quick first run: one magnification (200x), 3 seeds, 5 epochs → about 10–20 minutes on a T4.
Both split types use the **same model and settings**; only the split changes.

In [ ]:
!PYTHONPATH=/content/breakhis-reliability/src python -m breakhis.experiments.leakage --data_dir "$DATA_DIR" --out_dir "$RESULTS/leakage_200x" \
    --magnification 200 --seeds 0 1 2 --epochs 5

## 5. Results

In [ ]:
import pandas as pd
res = pd.read_csv(f'{RESULTS}/leakage_200x/leakage_results.csv')
cols = ['split','seed','patients_in_both','accuracy','balanced_accuracy','bal_acc_ci_low','bal_acc_ci_high','auroc','patient_balanced_accuracy','n_test_patients']
display(res[cols].round(3))
display(res.groupby('split')[['accuracy','balanced_accuracy','auroc','patient_balanced_accuracy']].agg(['mean','std']).round(3))

## 6. What to send back to Claude
Copy the two tables above into the chat. We'll read them together:
- How big is the gap between `image` and `patient` splits?
- How wide are the confidence intervals (only ~16 test patients per fold)?
- Does patient-level accuracy differ from image-level?